# Trabalho Prático 2 — Realidade Aumentada
**Introdução à Computação Visual — DCC/UFMG**

**Aluno(a):** Lucas Affonso Pires

**Vídeo de apresentação (YouTube):** https://www.youtube.com/watch?v=svi6emBa5lE

---

## Visão geral

O objetivo é detectar os alvos em cada frame de `video-entrada.mp4` e inserir, acima de cada um, um cubo e o modelo 3D do Pikachu. O notebook segue os passos do enunciado, na mesma ordem:

| Passo do enunciado | Onde está no notebook |
|---|---|
| 1. Calibrar a câmera com o tabuleiro e obter os parâmetros intrínsecos | `calibrar_camera` |
| 2. Detectar os alvos e localizar sua posição e orientação por casamento de templates | `DetectorAlvos` |
| 3. Obter a pose da câmera com `solvePnP` | `estimar_pose` |
| 4. Renderizar um cubo sobre cada alvo, mostrando sua orientação, só com OpenGL | `Scene.desenhar_cubo` |
| 5. Renderizar o Pikachu girando em torno de Z: anti-horário nos alvos 1 e 3, horário no 2 | `Scene.desenhar_pikachu` + `IdentificadorAlvos` |
| 6. Mostrar os alvos detectados como na Figura 2-a | `Scene.desenhar_deteccoes` |

A estrutura do código é a do **código esqueleto**: a classe `Scene` com `display()`, as funções `camera_config`, `initLighting`, `idleCallback` e `keyboard_callback`, e o bloco principal com `glutMainLoop()`.

**Arquivos necessários na mesma pasta:** `video-entrada.mp4`, `alvo.jpg`, `objloader.py` e a pasta `pikachu_obj/`. O resultado é salvo em `video-resultado.mp4`.

## Instalação das bibliotecas

Instala no ambiente do próprio kernel do Jupyter as bibliotecas usadas pelo trabalho: OpenCV, NumPy, PyOpenGL (que já traz o GLUT para Windows), PyGame (usado pelo `objloader.py` para ler as texturas) e Pillow. Se já estiverem instaladas, o comando não altera nada.

In [31]:
import os
import time
from itertools import permutations

import cv2
import numpy as np

from OpenGL.GL import *
from OpenGL.GLUT import *
from OpenGL.GLU import *

from PIL import Image

from objloader import *

from IPython.display import display, Image as IPyImage

VIDEO_ENTRADA = "video-entrada.mp4"
VIDEO_SAIDA = "video-resultado.mp4"
IMAGEM_ALVO = "alvo.jpg"
MODELO_PIKACHU = "./pikachu_obj/Pikachu.obj"


def mostrar(img_bgr, largura=None):
    if largura is not None:
        h, w = img_bgr.shape[:2]
        img_bgr = cv2.resize(img_bgr, (largura, int(h * largura / w)))
    display(IPyImage(data=cv2.imencode(".png", img_bgr)[1].tobytes()))

## Passo 1 — Calibração da câmera

### `pontos_tabuleiro` e `calibrar_camera`

**O que fazem.** `pontos_tabuleiro` gera as coordenadas 3D das quinas internas do tabuleiro, em centímetros, no plano $Z = 0$. `calibrar_camera` percorre o vídeo, encontra as quinas do tabuleiro nos frames e chama `cv2.calibrateCamera`, que devolve a matriz de parâmetros intrínsecos $K$ e os coeficientes de distorção da lente.

**Decisões.**

- **Ferramenta.** O enunciado permite usar o toolbox de Bouguet, mas não obriga. Usei a calibração da OpenCV, que segue o mesmo procedimento (fotos de um alvo plano conhecido) e mantém todo o trabalho no notebook.
- **Tamanho do padrão.** O tabuleiro tem 9 × 7 quadrados; a OpenCV procura as **quinas internas**, que são 8 × 6. Cada quadrado mede **3 cm** (dado do enunciado), então tudo fica em centímetros.
- **Frames usados.** Um a cada 10. Frames vizinhos são quase iguais, então não é preciso usar todos.
- **`cornerSubPix`.** Refina a posição das quinas encontradas, como no tutorial de calibração da OpenCV.

In [32]:
PADRAO_TABULEIRO = (8, 6)      # quinas internas (o tabuleiro tem 9 x 7 quadrados)
TAM_QUADRADO_CM = 3.0


def pontos_tabuleiro(padrao=PADRAO_TABULEIRO, tam=TAM_QUADRADO_CM):
    objp = np.zeros((padrao[0] * padrao[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:padrao[0], 0:padrao[1]].T.reshape(-1, 2) * tam
    return objp


def calibrar_camera(caminho_video, padrao=PADRAO_TABULEIRO, tam=TAM_QUADRADO_CM, passo=10):
    objp = pontos_tabuleiro(padrao, tam)
    criterio = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-3)
    pts_obj, pts_img = [], []
    cap = cv2.VideoCapture(caminho_video)
    tamanho, i = None, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % passo == 0:
            cinza = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            tamanho = cinza.shape[::-1]
            achou, cantos = cv2.findChessboardCorners(cinza, padrao)
            if achou:
                cantos = cv2.cornerSubPix(cinza, cantos, (11, 11), (-1, -1), criterio)
                pts_obj.append(objp)
                pts_img.append(cantos)
        i += 1
    cap.release()
    _, K, dist, _, _ = cv2.calibrateCamera(pts_obj, pts_img, tamanho, None, None)
    return K, dist

In [33]:
K, dist = calibrar_camera(VIDEO_ENTRADA)

np.set_printoptions(precision=4, suppress=True)
print("Matriz de parâmetros intrínsecos K:")
print(K)
print("\nCoeficientes de distorção [k1, k2, p1, p2, k3]:")
print(dist.ravel())

Matriz de parâmetros intrínsecos K:
[[416.3364   0.     315.622 ]
 [  0.     415.1631 229.7979]
 [  0.       0.       1.    ]]

Coeficientes de distorção [k1, k2, p1, p2, k3]:
[ 0.0732  0.0322 -0.0039  0.0004 -0.3962]


### Matriz de parâmetros intrínsecos estimada

$$
K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}
\approx
\begin{bmatrix} 416{,}34 & 0 & 315{,}62 \\ 0 & 415{,}16 & 229{,}80 \\ 0 & 0 & 1 \end{bmatrix}
$$

$f_x$ e $f_y$ são as distâncias focais em pixels (quase iguais, como esperado) e $(c_x, c_y)$ é o centro óptico, perto do centro da imagem de 640 × 480. Os coeficientes de distorção são passados ao `solvePnP` no passo 3.

## Passo 2 — Detecção e localização da orientação dos alvos

Seguindo o enunciado, **não** uso keypoints (o alvo não tem textura). O método tem duas partes: (1) achar quadriláteros na imagem com binarização e `findContours`; (2) confirmar cada um e descobrir sua orientação por **casamento de templates**.

### `criar_mascara_conteudo`

O vídeo tem **bordas pretas** em volta da imagem. Na binarização elas viram região escura e, quando um alvo encosta nelas, o contorno do alvo se junta ao da borda e deixa de ser um quadrilátero. A máscara marca a área com imagem para que a borda seja pintada de branco antes da binarização.

### Classe `DetectorAlvos`

- **`__init__`** — Lê `alvo.jpg`, reduz para 64 × 64, binariza e guarda o template nas **4 rotações** (0°, 90°, 180°, 270°).
- **`ordenar_cantos`** — Coloca os 4 vértices de um quadrilátero sempre na mesma ordem (sentido horário, começando pelo canto superior esquerdo), para a homografia levar o quadrilátero sempre ao mesmo quadrado.
- **`candidatos`** — Suaviza a imagem, binariza com o limiar de **Otsu** (calculado só na área com imagem), encontra os contornos com `findContours`, simplifica cada um com `approxPolyDP` e fica com os que têm **4 vértices** e são **convexos**, descartando os muito pequenos.
- **`casar_template`** — Calcula a **homografia** do quadrilátero para um quadrado de 64 × 64 (`getPerspectiveTransform` + `warpPerspective`), binariza esse recorte e compara com as 4 rotações do template usando `matchTemplate`. A rotação com maior nota indica a **orientação** do alvo.
- **`detectar`** — Aceita como alvo o quadrilátero com nota de pelo menos 0,7. Esse valor foi escolhido olhando os resultados: os alvos recebem nota alta, enquanto as casas do tabuleiro e outros quadriláteros da cena ficam bem abaixo. Em seguida, **gira a lista de cantos** conforme a rotação encontrada, para que o canto 0 seja sempre o canto superior esquerdo do template, e remove detecções repetidas do mesmo alvo.

Alvos cortados pela borda do quadro não formam um quadrilátero fechado e não são detectados: sem os 4 cantos não é possível calcular a pose.

In [34]:
def criar_mascara_conteudo(frame):
    cinza = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    x, y, w, h = cv2.boundingRect(cv2.findNonZero((cinza > 20).astype(np.uint8)))
    mascara = np.zeros_like(cinza)
    mascara[y + 2:y + h - 2, x + 2:x + w - 2] = 255
    return mascara


class DetectorAlvos:
    TAM = 64

    def __init__(self, caminho_alvo, limiar_score=0.7, area_min=300):
        alvo = cv2.imread(caminho_alvo, cv2.IMREAD_GRAYSCALE)
        alvo = cv2.resize(alvo, (self.TAM, self.TAM), interpolation=cv2.INTER_AREA)
        _, alvo = cv2.threshold(alvo, 127, 255, cv2.THRESH_BINARY)
        self.templates = [np.ascontiguousarray(np.rot90(alvo, k)) for k in range(4)]
        t = self.TAM - 1
        self.destino = np.array([[0, 0], [t, 0], [t, t], [0, t]], np.float32)
        self.limiar_score = limiar_score
        self.area_min = area_min

    @staticmethod
    def ordenar_cantos(p):
        centro = p.mean(axis=0)
        ang = np.arctan2(p[:, 1] - centro[1], p[:, 0] - centro[0])
        p = p[np.argsort(ang)]
        return np.roll(p, -np.argmin(p.sum(axis=1)), axis=0)

    def candidatos(self, cinza, mascara):
        suave = cv2.GaussianBlur(cinza, (5, 5), 0)
        suave[mascara == 0] = 255
        limiar, _ = cv2.threshold(suave[mascara > 0], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        binaria = (suave < limiar).astype(np.uint8) * 255
        contornos, _ = cv2.findContours(binaria, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        quads = []
        for c in contornos:
            if cv2.contourArea(c) < self.area_min:
                continue
            aprox = cv2.approxPolyDP(c, 0.03 * cv2.arcLength(c, True), True)
            if len(aprox) == 4 and cv2.isContourConvex(aprox):
                quads.append(self.ordenar_cantos(aprox.reshape(4, 2).astype(np.float32)))
        return quads

    def casar_template(self, cinza, quad):
        H = cv2.getPerspectiveTransform(quad, self.destino)
        recorte = cv2.warpPerspective(cinza, H, (self.TAM, self.TAM))
        _, recorte = cv2.threshold(recorte, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        scores = [float(cv2.matchTemplate(recorte, t, cv2.TM_CCOEFF_NORMED)[0, 0])
                  for t in self.templates]
        k = int(np.argmax(scores))
        return scores[k], k

    def detectar(self, cinza, mascara):
        achados = []
        for quad in self.candidatos(cinza, mascara):
            score, k = self.casar_template(cinza, quad)
            if score >= self.limiar_score:
                # se o recorte casou com o template girado k*90 graus (anti-horario),
                # o canto superior esquerdo do template esta na posicao (4 - k) % 4
                achados.append((score, np.roll(quad, -((4 - k) % 4), axis=0)))
        achados.sort(key=lambda a: -a[0])
        alvos = []
        for score, cantos in achados:
            if all(np.linalg.norm(cantos.mean(axis=0) - a["cantos"].mean(axis=0)) > 15 for a in alvos):
                alvos.append({"cantos": cantos, "score": score})
        return alvos

## Passo 3 — Pose da câmera com `solvePnP`

### `pontos_alvo` e `estimar_pose`

**Sistema de coordenadas do alvo.** Origem no centro do alvo, **X** para a direita e **Y** para cima do template, e **Z saindo do alvo**, para cima. Os 4 cantos, na mesma ordem do detector, são $(-L/2, L/2, 0)$, $(L/2, L/2, 0)$, $(L/2, -L/2, 0)$ e $(-L/2, -L/2, 0)$.

**Lado do alvo $L$ = 9 cm.** O enunciado não informa o tamanho do alvo, mas no vídeo ele tem aproximadamente o tamanho de **3 casas do tabuleiro** (3 × 3 cm). Usar centímetros deixa a pose dos alvos na mesma unidade da pose do tabuleiro, o que é usado na identificação dos alvos (abaixo).

**`estimar_pose`** chama `cv2.solvePnP` com os 4 cantos detectados, $K$ e os coeficientes de distorção, e devolve a rotação (`rvec`) e a translação (`tvec`) do alvo em relação à câmera:
- Na primeira vez que um alvo aparece, uso a opção `SOLVEPNP_IPPE_SQUARE`, própria para 4 pontos que formam um quadrado.
- Nos frames seguintes, passo a pose do frame anterior como ponto de partida (`useExtrinsicGuess=True`). Assistindo ao resultado, em alguns frames o cubo "tombava" de repente para outra inclinação e voltava no frame seguinte: um quadrado plano pode ser explicado por duas inclinações parecidas. Partindo da pose anterior, o `solvePnP` fica na inclinação certa (`estabilizar_poses`).

In [35]:
LADO_ALVO_CM = 9.0     # aproximadamente 3 casas do tabuleiro


def pontos_alvo(lado=LADO_ALVO_CM):
    m = lado / 2.0
    return np.array([[-m, m, 0], [m, m, 0], [m, -m, 0], [-m, -m, 0]], np.float32)


def estimar_pose(cantos, K, dist, chute=None, lado=LADO_ALVO_CM):
    obj = pontos_alvo(lado)
    img = cantos.astype(np.float32)
    if chute is None:
        ok, rvec, tvec = cv2.solvePnP(obj, img, K, dist, flags=cv2.SOLVEPNP_IPPE_SQUARE)
    else:
        ok, rvec, tvec = cv2.solvePnP(obj, img, K, dist, chute[0].copy(), chute[1].copy(),
                                      useExtrinsicGuess=True, flags=cv2.SOLVEPNP_ITERATIVE)
    return rvec, tvec


def estabilizar_poses(frames, K, dist, lacuna_max=5):
    anterior = {}
    for n, alvos in enumerate(frames):
        for a in alvos:
            prev = anterior.get(a["id"])
            chute = prev[1] if prev is not None and n - prev[0] <= lacuna_max else None
            a["rvec"], a["tvec"] = estimar_pose(a["cantos"], K, dist, chute)
            anterior[a["id"]] = (n, (a["rvec"], a["tvec"]))

## Identificação dos alvos 1, 2 e 3 (necessária para o passo 5)

O passo 5 pede sentidos de rotação diferentes para os alvos 1, 3 e 2, mas os três alvos são **iguais**, então o template não diz qual é qual. A ideia é usar o que se vê na cena (Figura 2-a): o alvo **1** fica ao lado do tabuleiro, o **2** em cima da caixa e o **3** sobre a mesa, mais longe. Como a cena não se mexe (só a câmera anda), essas posições valem para o vídeo todo.

1. **Frame de referência (`definir_referencia`).** No início do vídeo os três alvos e o tabuleiro aparecem juntos, como na Figura 2-a. No primeiro frame assim, numero os alvos pela posição na imagem: o mais à esquerda é o **1**; dos outros dois, o de cima é o **2** e o de baixo é o **3**. Para cada um, guardo a **distância até o centro do tabuleiro**.
2. **Pelo tabuleiro (`id_pelo_tabuleiro`).** Em qualquer frame em que o tabuleiro aparece, calculo a distância de cada alvo ao centro do tabuleiro e dou o número do alvo de referência com a distância mais parecida. Uso a distância, e não a direção, porque o tabuleiro é igual quando girado 180° e a OpenCV pode devolver suas quinas em ordem invertida.
3. **Pela posição entre alvos (`aprender_layout` / `id_pelo_layout`).** Quando o tabuleiro não aparece mas há dois ou mais alvos, uso a posição de um alvo em relação ao outro, que também não muda, aprendida nos frames já identificados.
4. **Pela continuidade (`montar_trilhas`).** Um alvo se move pouco de um frame para o seguinte. Ligo as detecções próximas em frames seguidos numa "trilha" e dou a toda a trilha o número que mais apareceu nela. Assim, um alvo que aparece sozinho e sem o tabuleiro recebe o número de quando foi identificado antes ou depois.

In [36]:
class IdentificadorAlvos:
    CENTRO_TABULEIRO = np.array([3.5 * TAM_QUADRADO_CM, 2.5 * TAM_QUADRADO_CM])

    def __init__(self, K, dist, raio_rastreio_px=60, lacuna_max=8, tolerancia_layout_cm=6.0):
        self.K, self.dist = K, dist
        self.objp = pontos_tabuleiro()
        self.raio = raio_rastreio_px
        self.lacuna_max = lacuna_max
        self.tolerancia_layout = tolerancia_layout_cm
        self.referencia = None
        self.layout = {}

    def pose_tabuleiro(self, cinza):
        achou, cantos = cv2.findChessboardCorners(cinza, PADRAO_TABULEIRO)
        if not achou:
            return None
        _, rvec, tvec = cv2.solvePnP(self.objp, cantos, self.K, self.dist)
        return cv2.Rodrigues(rvec)[0], tvec.ravel()

    def distancia_ao_tabuleiro(self, alvo, pose_tab):
        R_tab, t_tab = pose_tab
        X = R_tab.T @ (alvo["tvec"].ravel() - t_tab)
        return float(np.linalg.norm(X[:2] - self.CENTRO_TABULEIRO))

    def definir_referencia(self, alvos, pose_tab):
        alvos = sorted(alvos, key=lambda a: a["cantos"][:, 0].mean())
        um = alvos[0]
        dois, tres = sorted(alvos[1:], key=lambda a: a["cantos"][:, 1].mean())
        self.referencia = {i: self.distancia_ao_tabuleiro(a, pose_tab) for i, a in ((1, um), (2, dois), (3, tres))}

    def id_pelo_tabuleiro(self, alvo, pose_tab):
        d = self.distancia_ao_tabuleiro(alvo, pose_tab)
        return min(self.referencia, key=lambda i: abs(self.referencia[i] - d))

    @staticmethod
    def posicao_relativa(a, b):
        R_a = cv2.Rodrigues(a["rvec"])[0]
        return R_a.T @ (b["tvec"].ravel() - a["tvec"].ravel())

    def aprender_layout(self, frames):
        amostras = {}
        for alvos in frames:
            for a in alvos:
                for b in alvos:
                    if a is b or a["id_tab"] is None or b["id_tab"] is None or a["id_tab"] == b["id_tab"]:
                        continue
                    amostras.setdefault((a["id_tab"], b["id_tab"]), []).append(self.posicao_relativa(a, b))
        self.layout = {par: np.median(v, axis=0) for par, v in amostras.items()}

    def id_pelo_layout(self, alvos):
        if len(alvos) < 2:
            return
        melhor, custo_min = None, np.inf
        for perm in permutations([1, 2, 3], len(alvos)):
            if any(a["id_tab"] is not None and a["id_tab"] != p for a, p in zip(alvos, perm)):
                continue
            custos = []
            for (a, pa) in zip(alvos, perm):
                for (b, pb) in zip(alvos, perm):
                    if a is not b and (pa, pb) in self.layout:
                        custos.append(np.linalg.norm(self.posicao_relativa(a, b) - self.layout[(pa, pb)]))
            if custos and max(custos) < self.tolerancia_layout and np.mean(custos) < custo_min:
                melhor, custo_min = perm, np.mean(custos)
        if melhor is not None:
            for a, p in zip(alvos, melhor):
                a["id_layout"] = p

    def montar_trilhas(self, frames):
        trilhas, ativas = [], []
        for n, alvos in enumerate(frames):
            ativas = [t for t in ativas if n - t["ultimo"] <= self.lacuna_max]
            pares = sorted(((np.linalg.norm(a["cantos"].mean(0) - t["centro"]), i, j)
                            for i, a in enumerate(alvos) for j, t in enumerate(ativas)), key=lambda x: x[0])
            usados_a, usados_t = set(), set()
            for d, i, j in pares:
                if d > self.raio or i in usados_a or j in usados_t:
                    continue
                usados_a.add(i)
                usados_t.add(j)
                ativas[j]["membros"].append(alvos[i])
                ativas[j]["centro"], ativas[j]["ultimo"] = alvos[i]["cantos"].mean(0), n
            for i, a in enumerate(alvos):
                if i not in usados_a:
                    t = {"membros": [a], "centro": a["cantos"].mean(0), "ultimo": n}
                    trilhas.append(t)
                    ativas.append(t)
        return trilhas

    def identificar(self, frames, poses_tab):
        for alvos, pose_tab in zip(frames, poses_tab):
            if self.referencia is None and len(alvos) == 3 and pose_tab is not None:
                self.definir_referencia(alvos, pose_tab)
        for alvos, pose_tab in zip(frames, poses_tab):
            for a in alvos:
                a["id_tab"] = self.id_pelo_tabuleiro(a, pose_tab) if pose_tab is not None else None
                a["id_layout"] = None
        self.aprender_layout(frames)
        for alvos in frames:
            self.id_pelo_layout(alvos)
        for t in self.montar_trilhas(frames):
            votos = [a["id_tab"] for a in t["membros"] if a["id_tab"] is not None]
            votos += [a["id_layout"] for a in t["membros"] if a["id_layout"] is not None]
            ident = max(set(votos), key=votos.count) if votos else None
            for a in t["membros"]:
                a["id"] = ident

### `analisar_video`

Percorre o vídeo uma vez e, em cada frame, detecta os alvos (passo 2), calcula a pose de cada um (passo 3) e a do tabuleiro. Depois numera os alvos e estabiliza as poses. O resultado é uma lista, com um item por frame, que a renderização usa.

A análise é feita **antes** da exibição porque a numeração pela continuidade precisa olhar frames anteriores e posteriores. Assim o `display()` do OpenGL só precisa desenhar.

In [37]:
def analisar_video(caminho_video, K, dist, detector, identificador):
    cap = cv2.VideoCapture(caminho_video)
    frames, poses_tab, mascara = [], [], None
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if mascara is None:
            mascara = criar_mascara_conteudo(frame)
        cinza = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        alvos = detector.detectar(cinza, mascara)
        for a in alvos:
            a["rvec"], a["tvec"] = estimar_pose(a["cantos"], K, dist)
        poses_tab.append(identificador.pose_tabuleiro(cinza) if alvos else None)
        frames.append(alvos)
    cap.release()
    identificador.identificar(frames, poses_tab)
    estabilizar_poses(frames, K, dist)
    return frames

In [38]:
resultados = analisar_video(VIDEO_ENTRADA, K, dist, DetectorAlvos(IMAGEM_ALVO), IdentificadorAlvos(K, dist))

todas = [a for f in resultados for a in f]
print(f"Frames analisados: {len(resultados)}")
print(f"Alvos detectados no vídeo: {len(todas)}")
for i in (1, 2, 3):
    print(f"  alvo {i}: em {sum(a['id'] == i for a in todas)} frames")

Frames analisados: 1100
Alvos detectados no vídeo: 1004
  alvo 1: em 621 frames
  alvo 2: em 160 frames
  alvo 3: em 223 frames


## Passos 4, 5 e 6 — Renderização com OpenGL

Tudo o que aparece na cena aumentada é desenhado com OpenGL: o frame do vídeo como fundo, os contornos e números dos alvos, o cubo, o cone que mostra a orientação e o Pikachu. A OpenCV só lê o vídeo e salva o resultado.

### `matriz_projecao` — parâmetros intrínsecos → OpenGL

O esqueleto usa `gluPerspective`, que só recebe um ângulo de visão e supõe o centro óptico no meio da imagem. Para os objetos ficarem exatamente em cima dos alvos, a matriz de projeção é montada a partir de $K$:

$$
P = \begin{bmatrix}
\frac{2f_x}{W} & 0 & 1-\frac{2c_x}{W} & 0\\
0 & \frac{2f_y}{H} & \frac{2c_y}{H}-1 & 0\\
0 & 0 & -\frac{f+n}{f-n} & -\frac{2fn}{f-n}\\
0 & 0 & -1 & 0
\end{bmatrix}
$$

$W \times H$ é o tamanho da imagem e $n$, $f$ são os planos de recorte próximo e distante. Os sinais compensam que, na imagem, a linha cresce para baixo e, no OpenGL, o eixo y cresce para cima. A matriz é transposta antes do `glLoadMatrixd` porque o OpenGL lê matrizes por colunas.

### `matriz_modelview` — pose do `solvePnP` → OpenGL

Na OpenCV a câmera tem Y para baixo e olha para +Z; no OpenGL tem Y para cima e olha para −Z. Basta inverter os eixos Y e Z:

$$ M_{\text{modelview}} = \mathrm{diag}(1,-1,-1,1)\,\begin{bmatrix} R & t \\ 0 & 1 \end{bmatrix} $$

Com isso, desenhar no sistema de coordenadas do alvo (Z para cima) é só carregar essa matriz.

In [39]:
def matriz_projecao(K, largura, altura, perto=1.0, longe=1000.0):
    fx, fy, cx, cy = K[0, 0], K[1, 1], K[0, 2], K[1, 2]
    P = np.array([
        [2 * fx / largura, 0, 1 - 2 * cx / largura, 0],
        [0, 2 * fy / altura, 2 * cy / altura - 1, 0],
        [0, 0, -(longe + perto) / (longe - perto), -2 * longe * perto / (longe - perto)],
        [0, 0, -1, 0]], dtype=np.float64)
    return P.T.flatten()


CV_PARA_GL = np.diag([1.0, -1.0, -1.0, 1.0])


def matriz_modelview(rvec, tvec):
    R, _ = cv2.Rodrigues(rvec)
    M = np.eye(4)
    M[:3, :3] = R
    M[:3, 3] = tvec.ravel()
    return (CV_PARA_GL @ M).T.flatten()

### Classe `Scene` (estrutura do código esqueleto)

- **`__init__`** — Como no esqueleto, carrega o Pikachu com `OBJ(..., swapyz=True)` e inicializa `angulo_animacao`. Também abre o vídeo, guarda $K$ e os resultados da análise, e cria a textura do fundo e o arquivo de saída.
  - **Escala do Pikachu:** o modelo tem cerca de 4,7 unidades de altura; escalo para ele caber em pé dentro do cubo.
  - **Sentido de rotação (passo 5):** `{1: +1, 2: -1, 3: +1}`. Como o eixo Z do alvo aponta para cima, um ângulo positivo em `glRotatef(..., 0, 0, 1)` gira no sentido **anti-horário** visto de cima (alvos 1 e 3) e um negativo no **horário** (alvo 2).
  - O `swapyz=True` deixa o "para cima" do Pikachu no eixo Z, que é a normal do alvo, então ele fica em pé sem outras rotações.
- **`proximo_frame`** — Lê o próximo frame e pega os alvos já analisados desse frame.
- **`desenhar_fundo`** — Envia o frame como textura e desenha um retângulo cobrindo a janela, em projeção ortográfica, sem iluminação e sem teste de profundidade.
- **`desenhar_deteccoes` (passo 6)** — Como na Figura 2-a: contorno verde de cada alvo (`GL_LINE_LOOP`), um ponto no canto superior esquerdo do template e o número do alvo (`glutBitmapCharacter`). O número é vermelho para aparecer tanto no papel branco quanto na borda preta.
- **`desenhar_cubo` (passo 4)** — Cubo de arame (`glutWireCube`) com o lado do alvo, subido de $L/2$ em Z para ficar **apoiado sobre** o alvo. A **orientação** aparece num cone de arame (`glutWireCone`) saindo da face que aponta para o "topo" do template. O cubo não gira.
- **`desenhar_pikachu` (passo 5)** — Liga iluminação e textura, gira em torno de Z com o sentido do alvo, aplica a escala e desenha o modelo com `glCallList(self.obj3d.gl_list)`.
- **`display`** — Callback do GLUT: avança um frame; desenha o fundo e as detecções; carrega a projeção de $K$ e, para cada alvo, a modelview da sua pose, e desenha o cubo e o Pikachu; aumenta o ângulo da animação em 4° por frame; grava a imagem da janela e troca os buffers. Se o processamento for mais rápido que o vídeo, espera o tempo de um frame para a janela tocar na velocidade normal.
- **`gravar_frame` / `finalizar`** — `glReadPixels` lê a imagem renderizada (que vem de cabeça para baixo, por isso o `flip`) e o `cv2.VideoWriter` a grava. No fim do vídeo, ou com ESC/q, o arquivo é fechado, a janela é destruída e o laço do GLUT termina com `glutLeaveMainLoop()`, devolvendo o controle ao notebook. O `os._exit` do esqueleto fecharia o próprio Jupyter.

In [40]:
class Scene:
    def __init__(self, obj3d_filename, K, dist, caminho_video, resultados, caminho_saida=None, tempo_real=True):

        # Carrega o modelo 3D do Pikachu
        self.obj3d = OBJ(obj3d_filename, swapyz=True)

        self.angulo_animacao = 0
        self.passo_animacao = 4.0

        self.K, self.dist = K, dist
        self.cap = cv2.VideoCapture(caminho_video)
        self.largura = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.altura = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.fps = self.cap.get(cv2.CAP_PROP_FPS) or 30.0
        self.resultados = resultados
        self.n_frame = 0
        self.alvos = []
        self.frame = None
        self.janela = (self.largura, self.altura)
        self.tempo_real = tempo_real
        self.t_ultimo = None

        self.escala_pikachu = 0.95 * LADO_ALVO_CM / 4.72
        self.sentido = {1: +1, 2: -1, 3: +1}   # +1 anti-horario, -1 horario

        self.textura_fundo = glGenTextures(1)
        glBindTexture(GL_TEXTURE_2D, self.textura_fundo)
        glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
        glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)

        self.writer = None
        if caminho_saida:
            self.writer = cv2.VideoWriter(caminho_saida, cv2.VideoWriter_fourcc(*"mp4v"),
                                          self.fps, (self.largura, self.altura))
        self.finalizado = False
        self.id_janela = None

    def proximo_frame(self):
        ok, frame = self.cap.read()
        if not ok:
            return False
        self.frame = frame
        self.alvos = self.resultados[self.n_frame] if self.n_frame < len(self.resultados) else []
        self.n_frame += 1
        return True

    def desenhar_fundo(self):
        glDisable(GL_DEPTH_TEST)
        glDisable(GL_LIGHTING)
        glEnable(GL_TEXTURE_2D)
        glMatrixMode(GL_PROJECTION)
        glLoadIdentity()
        glOrtho(0, self.largura, self.altura, 0, -1, 1)
        glMatrixMode(GL_MODELVIEW)
        glLoadIdentity()
        rgb = cv2.cvtColor(self.frame, cv2.COLOR_BGR2RGB)
        glBindTexture(GL_TEXTURE_2D, self.textura_fundo)
        glPixelStorei(GL_UNPACK_ALIGNMENT, 1)
        glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, self.largura, self.altura, 0,
                     GL_RGB, GL_UNSIGNED_BYTE, rgb)
        glColor3f(1, 1, 1)
        glBegin(GL_QUADS)
        glTexCoord2f(0, 0); glVertex2f(0, 0)
        glTexCoord2f(1, 0); glVertex2f(self.largura, 0)
        glTexCoord2f(1, 1); glVertex2f(self.largura, self.altura)
        glTexCoord2f(0, 1); glVertex2f(0, self.altura)
        glEnd()
        glDisable(GL_TEXTURE_2D)

    def desenhar_deteccoes(self):
        glLineWidth(3.0)
        for a in self.alvos:
            glColor3f(0.0, 1.0, 0.0)
            glBegin(GL_LINE_LOOP)
            for (u, v) in a["cantos"]:
                glVertex2f(u, v)
            glEnd()
            glColor3f(1.0, 0.0, 0.0)
            glPointSize(7.0)
            glBegin(GL_POINTS)
            glVertex2f(*a["cantos"][0])
            glEnd()
            if a["id"] is not None:
                umin = a["cantos"][:, 0].min()
                vmed = a["cantos"][:, 1].mean()
                glRasterPos2f(max(umin - 18, 2), vmed + 6)
                for ch in str(a["id"]):
                    glutBitmapCharacter(GLUT_BITMAP_HELVETICA_18, ord(ch))
        glLineWidth(1.0)

    def desenhar_cubo(self):
        L = LADO_ALVO_CM
        glDisable(GL_LIGHTING)
        glDisable(GL_TEXTURE_2D)
        glColor3f(1.0, 1.0, 1.0)
        glLineWidth(1.5)
        glPushMatrix()
        glTranslatef(0, 0, L / 2)          # apoia o cubo sobre o alvo
        glutWireCube(L)
        glPopMatrix()
        glPushMatrix()
        glTranslatef(0, L / 2, L / 2)      # centro da face +Y (topo do template)
        glRotatef(-90, 1, 0, 0)            # o eixo do cone (+Z) passa a apontar para +Y
        glutWireCone(0.25 * L, 0.6 * L, 12, 3)
        glPopMatrix()
        glLineWidth(1.0)

    def desenhar_pikachu(self, ident):
        glEnable(GL_LIGHTING)
        glEnable(GL_TEXTURE_2D)
        glColor3f(1.0, 1.0, 1.0)
        glPushMatrix()
        # Rotaciona o objeto em torno do eixo Z (normal do alvo) para criar a animacao
        glRotatef(self.sentido.get(ident, +1) * self.angulo_animacao, 0, 0, 1)
        glScalef(self.escala_pikachu, self.escala_pikachu, self.escala_pikachu)
        # Renderiza o modelo 3D
        glCallList(self.obj3d.gl_list)
        glPopMatrix()
        glDisable(GL_TEXTURE_2D)
        glDisable(GL_LIGHTING)

    def display(self):
        if self.finalizado:
            return
        if not self.proximo_frame():
            self.finalizar()
            return

        glViewport(0, 0, *self.janela)
        glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

        # fundo (frame do video) e alvos detectados, em coordenadas de pixel
        self.desenhar_fundo()
        self.desenhar_deteccoes()

        # objetos 3D: projecao a partir de K e modelview a partir da pose de cada alvo
        glEnable(GL_DEPTH_TEST)
        glClear(GL_DEPTH_BUFFER_BIT)
        glMatrixMode(GL_PROJECTION)
        glLoadMatrixd(matriz_projecao(self.K, self.largura, self.altura))
        for a in self.alvos:
            if a["id"] is None:
                continue
            glMatrixMode(GL_MODELVIEW)
            glLoadMatrixd(matriz_modelview(a["rvec"], a["tvec"]))
            self.desenhar_cubo()
            self.desenhar_pikachu(a["id"])

        self.angulo_animacao = (self.angulo_animacao + self.passo_animacao) % 360

        if self.writer is not None:
            self.gravar_frame()

        if self.tempo_real:
            agora = time.time()
            if self.t_ultimo is not None:
                espera = 1.0 / self.fps - (agora - self.t_ultimo)
                if espera > 0:
                    time.sleep(espera)
            self.t_ultimo = time.time()

        glutSwapBuffers()

    def gravar_frame(self):
        w, h = self.janela
        glPixelStorei(GL_PACK_ALIGNMENT, 1)
        dados = glReadPixels(0, 0, w, h, GL_BGR, GL_UNSIGNED_BYTE)
        img = np.frombuffer(dados, np.uint8).reshape(h, w, 3)
        img = cv2.flip(img, 0)
        if (w, h) != (self.largura, self.altura):
            img = cv2.resize(img, (self.largura, self.altura))
        self.writer.write(img)

    def finalizar(self):
        if self.finalizado:
            return
        self.finalizado = True
        self.cap.release()
        if self.writer is not None:
            self.writer.release()
        print(f"{self.n_frame} frames renderizados.")
        try:
            if glutGet(GLUT_INIT_STATE):
                if self.id_janela is not None:
                    glutDestroyWindow(self.id_janela)
                glutLeaveMainLoop()
        except Exception:
            os._exit(0)

### Funções do esqueleto: `camera_config`, `initLighting`, `idleCallback`, `keyboard_callback`

- **`camera_config`**: mantém o *viewport* do tamanho da janela. A projeção não é mais o `gluPerspective`; ela vem de $K$ e é carregada em `Scene.display`.
- **`initLighting`**: mesma estrutura do esqueleto, com luz direcional e luz ambiente um pouco mais forte, para o lado do Pikachu oposto à luz não ficar escuro.
- **`idleCallback`**: igual ao esqueleto.
- **`keyboard_callback`**: ESC ou `q` encerram, fechando o vídeo de saída corretamente.

In [41]:
scene = None


def camera_config(width, height):

    glClearColor(0.2, 0.2, 0.2, 0.0)
    glClearDepth(1.0)

    glEnable(GL_DEPTH_TEST)

    # Os parametros intrinsecos da camera sao aplicados em Scene.display (matriz_projecao)
    glViewport(0, 0, width, height)
    if scene is not None:
        scene.janela = (width, height)


def initLighting():

    glEnable(GL_LIGHTING)
    glEnable(GL_LIGHT0)
    glEnable(GL_COLOR_MATERIAL)

    light_pos = [1.0, 1.0, 1.0, 0.0]
    glLightfv(GL_LIGHT0, GL_POSITION, light_pos)

    diffuse = [1.0, 1.0, 1.0, 1.0]
    ambient = [0.4, 0.4, 0.4, 1.0]
    specular = [1.0, 1.0, 1.0, 1.0]
    glLightfv(GL_LIGHT0, GL_DIFFUSE, diffuse)
    glLightfv(GL_LIGHT0, GL_AMBIENT, ambient)
    glLightfv(GL_LIGHT0, GL_SPECULAR, specular)

    glShadeModel(GL_SMOOTH)


def idleCallback():

    glutPostRedisplay()


def keyboard_callback(key, x, y):
    if key == b'\x1b' or key == b'q':  # ESC or 'q' to exit
        print("Exiting application...")
        scene.finalizar()

### Execução (bloco principal do esqueleto)

Abre uma janela GLUT de 640 × 480 (o tamanho do vídeo), cria a `Scene` **depois** da janela (o modelo e as texturas precisam de um contexto OpenGL ativo) e entra no `glutMainLoop()`. A janela toca o vídeo com a realidade aumentada, o resultado é gravado em `video-resultado.mp4` e, no fim, a janela fecha sozinha.

In [42]:
if __name__ == '__main__':

    # o freeglut se desliga ao sair do glutMainLoop; para poder executar esta celula
    # de novo no mesmo kernel, ele e inicializado outra vez quando necessario
    import OpenGL.GLUT.special as glut_especial
    if not glutGet(GLUT_INIT_STATE):
        glut_especial.INITIALIZED = False
        glutInit()
    glutInitDisplayMode(GLUT_RGBA | GLUT_DOUBLE | GLUT_DEPTH)
    glutInitWindowSize(640, 480)
    janela = glutCreateWindow(b'Realidade Aumentada - TP2')
    glutSetOption(GLUT_ACTION_ON_WINDOW_CLOSE, GLUT_ACTION_CONTINUE_EXECUTION)

    initLighting()

    scene = Scene(MODELO_PIKACHU, K, dist, VIDEO_ENTRADA, resultados, VIDEO_SAIDA)
    scene.id_janela = janela
    glutDisplayFunc(scene.display)

    glutReshapeFunc(camera_config)
    glutIdleFunc(idleCallback)
    glutKeyboardFunc(keyboard_callback)
    glutMainLoop()

    scene.finalizar()

1100 frames renderizados.


## Observações

- Alvos cortados pela borda do quadro não são detectados, porque não têm os 4 cantos visíveis; nesses frames não aparece objeto sobre eles.
- A numeração dos alvos depende de a cena ficar parada, o que acontece neste vídeo: só a câmera se move.

## Referências

- J.-Y. Bouguet. *Camera Calibration Toolbox for Matlab* — http://www.vision.caltech.edu/bouguetj/calib_doc/
- Documentação da OpenCV: calibração de câmera (`calibrateCamera`, `findChessboardCorners`, `cornerSubPix`), `solvePnP`, `findContours`, `approxPolyDP`, `getPerspectiveTransform`, `matchTemplate` — https://docs.opencv.org/
- Documentação do PyOpenGL e do freeglut (`glutWireCube`, `glutWireCone`, `glutBitmapCharacter`, `glutLeaveMainLoop`).
- Material da disciplina: código esqueleto, `objloader.py`, modelo do Pikachu e o vídeo indicado no enunciado (https://youtu.be/1z0Sga8_RxE).
- Assistência de IA: Claude (Anthropic) foi usado como apoio no desenvolvimento e na redação deste notebook.